# Week 05 — Machine learning foundations for seismic attributes

**Seismic Python & AI Engineering | Hands-on notebook**

Build a reproducible supervised classification workflow on *toy* seismic-like attributes. The emphasis is on good data handling, not high scores.

**Estimated independent study:** 4–5 hours in this notebook, plus 2–3 hours on repository exercises and review.  
**Data:** entirely synthetic; no external API, proprietary dataset, or GPU required.  
**Prerequisite:** complete the root README setup and open Jupyter from the repository root.

### Learning outcomes
- Identify features, labels, groups and potential leakage.
- Build an sklearn Pipeline with training-only preprocessing.
- Evaluate baseline classification with appropriate metrics.
- Explain why synthetic facies labels are not geological evidence.

### How to study
Read the explanations, execute the worked cells in order, complete the **Your turn** tasks, and implement the week's function in `src/seismic_toolkit/challenges.py`. Run the targeted exercise test to assess progress. The initial baseline tests should pass even before you complete the challenges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from seismic_toolkit import __version__
print(f"Seismic toolkit version: {__version__}")
np.set_printoptions(precision=4, suppress=True)

## 1. Problem definition and synthetic data
We will classify a simulated `facies` label from four invented seismic-like attributes: impedance, amplitude, frequency and coherence. The generator includes a `group` column standing in for correlated samples from a survey/location.

**Warning:** these samples do **not** reproduce the complexities of reservoir geology. Scores measure toy-pattern learning, not industrial interpretation quality.

In [ ]:
from seismic_toolkit.datasets import FEATURE_COLUMNS, make_synthetic_facies_dataset
frame = make_synthetic_facies_dataset(n_groups=12, samples_per_group=40, seed=42)
print(frame.head())
print("Rows:", len(frame), "groups:", frame.group.nunique(), "facies counts:", frame.facies.value_counts().to_dict())

## 2. Exploratory data analysis
Examine scale, missingness, class counts, outliers and distribution changes between groups. A feature may accidentally leak the answer if it encodes geological interpretation labels, sample IDs or future measurements.

**Your turn A:** plot two more features and compare the relative separability of classes. Write down **two reasons** why this visual separation may be misleading for real data.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for label, subgroup in frame.groupby("facies"):
    ax.hist(subgroup["impedance"], bins=20, alpha=0.4, label=f"facies {label}")
ax.set(xlabel="Simulated impedance (arbitrary units)", ylabel="Samples", title="Toy facies separation")
ax.legend()
plt.show()

## 3. Train/test isolation and sklearn Pipeline
A `Pipeline` ensures preprocessing steps are fitted on **training data only**, not on held-out test records. Start with a simple logistic regression baseline before trying complex models. For correlated survey samples, a randomly shuffled row split often gives optimistic validation; Week 6 treats this carefully.

In [ ]:
from seismic_toolkit.ml import make_baseline_model, evaluate_group_holdout
result = evaluate_group_holdout(frame, seed=10)
print({k: round(v, 3) for k, v in result.items() if k in {"accuracy", "balanced_accuracy", "macro_f1"}})
print("Train groups:", result["train_groups"])
print("Test groups:", result["test_groups"])
assert set(result["train_groups"]).isdisjoint(result["test_groups"])

## 4. Classification metrics
Accuracy alone can mislead on imbalanced classes. Balanced accuracy averages recall across classes; macro F1 gives each class equal weight regardless of its prevalence. A confusion matrix shows *which* classes are confused. Evaluate uncertainty over more than one group split before claiming robustness.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
print(classification_report(result["y_true"], result["y_pred"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(result["y_true"], result["y_pred"], cmap="Blues")
plt.title("Grouped holdout: synthetic labels")
plt.show()

## 5. Independent implementation — Week 5 challenge
Implement `fit_baseline_classifier(frame)` in `challenges.py`. It must:

- Fit a **scikit-learn Pipeline**, not a stand-alone estimator.
- Use the four `FEATURE_COLUMNS` and `facies` target.
- Avoid using `group` as a predictive feature.
- Include missing-value handling, scaling and an interpretable baseline classifier.
- Return the fitted pipeline and reject missing required columns.

**Your turn B:** fit a pipeline on a training subset and predict five held-out rows. Explain why fitting a scaler to all rows is a form of leakage.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=11)
train_idx, test_idx = next(splitter.split(frame[list(FEATURE_COLUMNS)], frame.facies, frame.group))
model = make_baseline_model()
model.fit(frame.iloc[train_idx][list(FEATURE_COLUMNS)], frame.iloc[train_idx].facies)
print("Example held-out predictions:", model.predict(frame.iloc[test_idx][list(FEATURE_COLUMNS)][:5]))

## 6. Applied problem — more accuracy is not always better
Compare logistic regression against a random forest on **the same group-separated split**. Report macro F1 and confusion matrices, not only train accuracy. Discuss model interpretability, overfitting risk and what additional field validation would be required before real-world deployment.

**Stretch:** add missing values to 3% of training features with a seeded RNG. Verify that the pipeline still works without imputing from test data.

---

## Weekly submission and self-review

1. Complete the independent exercise in `src/seismic_toolkit/challenges.py`.
2. Run: `python -m pytest -o addopts='' -m exercise -k week5 -q`.
3. Run baseline regression tests: `python -m pytest -q`.
4. Commit a focused change to a feature branch; document what changed and why.
5. Record one failure mode, one design decision, and one next-step question in your learning journal.

**Further work:** see [`../exercises/week05.md`](../exercises/week05.md) for a graded problem, extra practice and acceptance criteria. Compare with `solutions/reference_challenges.py` only **after** attempting your own solution.

**Important:** all datasets and algorithms are educational demonstrations, not validated reservoir-interpretation systems.